# Listas, tuplas y conjuntos · Relaciones binarias

Una relación binaria sobre un universo $U$ es un subconjunto de $U \times U$: en Python, un conjunto de pares `(x, y)` junto al conjunto `U`. El notebook la representa así y como matriz booleana, comprueba sus propiedades (reflexiva, simétrica, transitiva…), la compone consigo misma, calcula sus clausuras, las clases de equivalencia y los elementos extremos de un orden, y termina con el diagrama de Hasse. Las propiedades conocidas se comprueban con `assert` sobre relaciones aleatorias con semilla fija. Ideas de los ejercicios de [José A. Alonso](https://github.com/jaalonso/Ejercicios-Python).

## 1. Representar una relación

Representar tres relaciones como un universo y un conjunto de pares: la divisibilidad en $\{1, \dots, 12\}$ ($a$ se relaciona con $b$ si $a$ divide a $b$), «está en la misma isla que» entre municipios canarios y «hay una guagua directa de $a$ a $b$» entre pueblos de Gran Canaria. Escribir `es_relacion(u, r)`, que compruebe que todos los pares usan elementos del universo, e `imagen(r, x)`, el conjunto de elementos con los que se relaciona `x`.

In [1]:
import itertools
import random
from collections import Counter, defaultdict


def es_relacion(u, r):
    return all(x in u and y in u for x, y in r)


def imagen(r, x):
    return {y for a, y in r if a == x}


U_DIV = set(range(1, 13))
DIV = {(a, b) for a in U_DIV for b in U_DIV if b % a == 0}

ISLA = {
    "Las Palmas de GC": "Gran Canaria", "Telde": "Gran Canaria", "Agüimes": "Gran Canaria",
    "Santa Cruz": "Tenerife", "La Laguna": "Tenerife", "Adeje": "Tenerife",
    "Arrecife": "Lanzarote", "Puerto del Rosario": "Fuerteventura",
}
U_MUN = set(ISLA)
MISMA_ISLA = {(a, b) for a in U_MUN for b in U_MUN if ISLA[a] == ISLA[b]}

U_GUA = {"Las Palmas", "Telde", "Agüimes", "Arucas", "Gáldar", "Agaete"}
GUAGUA = {
    ("Las Palmas", "Telde"), ("Telde", "Las Palmas"), ("Telde", "Agüimes"), ("Agüimes", "Telde"),
    ("Las Palmas", "Arucas"), ("Arucas", "Gáldar"), ("Gáldar", "Agaete"), ("Agaete", "Gáldar"),
}

for nombre, u, r in [("Divisibilidad", U_DIV, DIV), ("Misma isla", U_MUN, MISMA_ISLA), ("Guagua directa", U_GUA, GUAGUA)]:
    print(f"{nombre:<15} |U| = {len(u):>2}   |R| = {len(r):>2}")
print("Múltiplos de 3:", sorted(imagen(DIV, 3)))
print("En la misma isla que Telde:", sorted(imagen(MISMA_ISLA, "Telde")))
print("Guaguas directas desde Telde:", sorted(imagen(GUAGUA, "Telde")))

assert all(es_relacion(u, r) for u, r in [(U_DIV, DIV), (U_MUN, MISMA_ISLA), (U_GUA, GUAGUA)])
assert not es_relacion(U_DIV, {(3, 13)})
assert len(DIV) == sum(12 // a for a in U_DIV)
assert imagen(DIV, 3) == {3, 6, 9, 12}

Divisibilidad   |U| = 12   |R| = 35
Misma isla      |U| =  8   |R| = 20
Guagua directa  |U| =  6   |R| =  8
Múltiplos de 3: [3, 6, 9, 12]
En la misma isla que Telde: ['Agüimes', 'Las Palmas de GC', 'Telde']
Guaguas directas desde Telde: ['Agüimes', 'Las Palmas']


## 2. Matriz booleana

Con los elementos del universo ordenados, una relación es una matriz cuadrada de booleanos cuya casilla $(i, j)$ vale `True` si el $i$-ésimo elemento se relaciona con el $j$-ésimo. Escribir `a_matriz`, `de_matriz` y una función que la muestre con `1` y `·`, y comprobar que al convertir ida y vuelta se recupera la misma relación.

In [2]:
def a_matriz(u, r):
    orden = sorted(u)
    return orden, [[(x, y) in r for y in orden] for x in orden]


def de_matriz(orden, m):
    return {(x, y) for x, fila in zip(orden, m) for y, v in zip(orden, fila) if v}


def mostrar_matriz(u, r, etiqueta=str, ancho=4):
    orden, m = a_matriz(u, r)
    print(" " * ancho + "".join(f"{etiqueta(y):>{ancho}}" for y in orden))
    for x, fila in zip(orden, m):
        print(f"{etiqueta(x):>{ancho}}" + "".join(f"{'1' if v else '·':>{ancho}}" for v in fila))


mostrar_matriz(U_DIV, DIV)
print()
mostrar_matriz(U_GUA, GUAGUA, etiqueta=lambda x: x[:3])

for u, r in [(U_DIV, DIV), (U_MUN, MISMA_ISLA), (U_GUA, GUAGUA)]:
    orden, m = a_matriz(u, r)
    assert de_matriz(orden, m) == r
    assert sum(map(sum, m)) == len(r)

       1   2   3   4   5   6   7   8   9  10  11  12
   1   1   1   1   1   1   1   1   1   1   1   1   1
   2   ·   1   ·   1   ·   1   ·   1   ·   1   ·   1
   3   ·   ·   1   ·   ·   1   ·   ·   1   ·   ·   1
   4   ·   ·   ·   1   ·   ·   ·   1   ·   ·   ·   1
   5   ·   ·   ·   ·   1   ·   ·   ·   ·   1   ·   ·
   6   ·   ·   ·   ·   ·   1   ·   ·   ·   ·   ·   1
   7   ·   ·   ·   ·   ·   ·   1   ·   ·   ·   ·   ·
   8   ·   ·   ·   ·   ·   ·   ·   1   ·   ·   ·   ·
   9   ·   ·   ·   ·   ·   ·   ·   ·   1   ·   ·   ·
  10   ·   ·   ·   ·   ·   ·   ·   ·   ·   1   ·   ·
  11   ·   ·   ·   ·   ·   ·   ·   ·   ·   ·   1   ·
  12   ·   ·   ·   ·   ·   ·   ·   ·   ·   ·   ·   1

     Aga Agü Aru Gál Las Tel
 Aga   ·   ·   ·   1   ·   ·
 Agü   ·   ·   ·   ·   ·   1
 Aru   ·   ·   ·   1   ·   ·
 Gál   1   ·   ·   ·   ·   ·
 Las   ·   ·   1   ·   ·   1
 Tel   ·   1   ·   ·   1   ·


## 3. Reflexiva e irreflexiva

Una relación es reflexiva si todo elemento se relaciona consigo mismo (la diagonal de la matriz está llena) e irreflexiva si ninguno lo hace (la diagonal está vacía). Escribir ambas funciones y otra que genere relaciones aleatorias. Comprobar que la divisibilidad es reflexiva, que «menor que» es irreflexiva, que la de las guaguas es irreflexiva (nadie coge una guagua para ir a su propio pueblo), que «el producto es par» no es ninguna de las dos y que, en un universo no vacío, ninguna relación puede ser ambas cosas a la vez.

In [3]:
def es_reflexiva(u, r):
    return all((x, x) in r for x in u)


def es_irreflexiva(u, r):
    return all((x, x) not in r for x in u)


def relacion_aleatoria(u, p, rng):
    return {(x, y) for x in u for y in u if rng.random() < p}


MENOR = {(a, b) for a in U_DIV for b in U_DIV if a < b}
PRODUCTO_PAR = {(a, b) for a in U_DIV for b in U_DIV if a * b % 2 == 0}
for nombre, u, r in [("Divisibilidad", U_DIV, DIV), ("Menor que", U_DIV, MENOR), ("Guagua directa", U_GUA, GUAGUA),
                     ("Producto par", U_DIV, PRODUCTO_PAR)]:
    print(f"{nombre:<15} reflexiva: {es_reflexiva(u, r)!s:<5}  irreflexiva: {es_irreflexiva(u, r)}")
print("Relación vacía en un universo vacío:", es_reflexiva(set(), set()), es_irreflexiva(set(), set()))

rng = random.Random(1)
for _ in range(1000):
    u = set(range(rng.randint(1, 6)))
    r = relacion_aleatoria(u, rng.random(), rng)
    assert not (es_reflexiva(u, r) and es_irreflexiva(u, r))
    orden, m = a_matriz(u, r)
    assert es_reflexiva(u, r) == all(m[i][i] for i in range(len(orden)))
    assert es_irreflexiva(u, r) == (not any(m[i][i] for i in range(len(orden))))

Divisibilidad   reflexiva: True   irreflexiva: False
Menor que       reflexiva: False  irreflexiva: True
Guagua directa  reflexiva: False  irreflexiva: True
Producto par    reflexiva: False  irreflexiva: False
Relación vacía en un universo vacío: True True


## 4. Simétrica, antisimétrica y asimétrica

Una relación es simétrica si $(x, y) \in R \Rightarrow (y, x) \in R$; antisimétrica si $(x, y)$ e $(y, x)$ solo están a la vez cuando $x = y$; y asimétrica si nunca están a la vez (ni siquiera con $x = y$). Escribir las tres funciones y la relación inversa $R^{-1} = \{(y, x) : (x, y) \in R\}$, y comprobar con relaciones aleatorias que:

- $R$ es simétrica si y solo si $R = R^{-1}$.
- $R$ es asimétrica si y solo si es antisimétrica e irreflexiva.
- $R$ es simétrica y antisimétrica a la vez si y solo si solo contiene pares $(x, x)$.

In [4]:
def inversa(r):
    return {(y, x) for x, y in r}


def es_simetrica(r):
    return all((y, x) in r for x, y in r)


def es_antisimetrica(r):
    return all(x == y or (y, x) not in r for x, y in r)


def es_asimetrica(r):
    return all((y, x) not in r for x, y in r)


for nombre, r in [("Divisibilidad", DIV), ("Menor que", MENOR), ("Misma isla", MISMA_ISLA), ("Guagua directa", GUAGUA)]:
    print(f"{nombre:<15} simétrica: {es_simetrica(r)!s:<5}  antisimétrica: {es_antisimetrica(r)!s:<5}  asimétrica: {es_asimetrica(r)}")

rng = random.Random(2)
for _ in range(2000):
    u = set(range(rng.randint(1, 5)))
    r = relacion_aleatoria(u, rng.random() * 0.6, rng)
    assert es_simetrica(r) == (r == inversa(r))
    assert es_asimetrica(r) == (es_antisimetrica(r) and es_irreflexiva(u, r))
    assert (es_simetrica(r) and es_antisimetrica(r)) == all(x == y for x, y in r)

Divisibilidad   simétrica: False  antisimétrica: True   asimétrica: False
Menor que       simétrica: False  antisimétrica: True   asimétrica: True
Misma isla      simétrica: True   antisimétrica: False  asimétrica: False
Guagua directa  simétrica: False  antisimétrica: False  asimétrica: False


## 5. Transitiva y total

$R$ es transitiva si de $(x, y) \in R$ e $(y, z) \in R$ se deduce $(x, z) \in R$, y total si dos elementos cualesquiera están relacionados en algún sentido. Escribir `es_transitiva` de dos formas: con un triple bucle y comparando conjuntos imagen ($y \in R(x) \Rightarrow R(y) \subseteq R(x)$), y `contraejemplos_transitiva`, que genere los tríos $(x, y, z)$ que la incumplen. Comprobar con relaciones aleatorias que ambas versiones coinciden y que toda relación total es reflexiva.

In [5]:
def es_transitiva_bruta(r):
    return all((x, z) in r for x, y in r for y2, z in r if y == y2)


def sucesores(r):
    suc = defaultdict(set)
    for x, y in r:
        suc[x].add(y)
    return suc


def es_transitiva(r):
    suc = sucesores(r)
    return all(suc[y] <= suc[x] for x, y in r)


def contraejemplos_transitiva(r):
    suc = sucesores(r)
    for x, y in sorted(r):
        for z in sorted(suc[y] - suc[x]):
            yield x, y, z


def es_total(u, r):
    return all((x, y) in r or (y, x) in r for x in u for y in u)


MENOR_IGUAL = MENOR | {(a, a) for a in U_DIV}
for nombre, u, r in [("Divisibilidad", U_DIV, DIV), ("Menor o igual", U_DIV, MENOR_IGUAL),
                     ("Misma isla", U_MUN, MISMA_ISLA), ("Guagua directa", U_GUA, GUAGUA)]:
    print(f"{nombre:<15} transitiva: {es_transitiva(r)!s:<5}  total: {es_total(u, r)}")
x, y, z = next(contraejemplos_transitiva(GUAGUA))
print(f"Guaguas: {x} → {y} → {z}, pero no {x} → {z}")
x, y, z = next((x, y, z) for x, y, z in contraejemplos_transitiva(GUAGUA) if x != z)
print(f"Guaguas: {x} → {y} → {z}, pero no {x} → {z}")
x, y = next((x, y) for x in sorted(U_DIV) for y in sorted(U_DIV) if (x, y) not in DIV and (y, x) not in DIV)
print(f"Divisibilidad: ni {x} divide a {y} ni {y} a {x}")

rng = random.Random(3)
for _ in range(2000):
    u = set(range(rng.randint(1, 5)))
    r = relacion_aleatoria(u, rng.random(), rng)
    assert es_transitiva(r) == es_transitiva_bruta(r)
    assert es_transitiva(r) == (next(contraejemplos_transitiva(r), None) is None)
    assert not es_total(u, r) or es_reflexiva(u, r)

Divisibilidad   transitiva: True   total: False
Menor o igual   transitiva: True   total: True
Misma isla      transitiva: True   total: False
Guagua directa  transitiva: False  total: False
Guaguas: Agaete → Gáldar → Agaete, pero no Agaete → Agaete
Guaguas: Agüimes → Telde → Las Palmas, pero no Agüimes → Las Palmas
Divisibilidad: ni 2 divide a 3 ni 3 a 2


## 6. Equivalencias y órdenes

Una relación es de equivalencia si es reflexiva, simétrica y transitiva; de orden parcial si es reflexiva, antisimétrica y transitiva; y de orden total si además es total. Clasificar las relaciones de ejemplo (añadiendo «mismo resto al dividir entre 3» y la inclusión entre los subconjuntos de $\{1, 2, 3\}$) y contar por fuerza bruta cuántas relaciones de cada tipo hay en un universo de $n \le 4$ elementos, entre las $2^{n^2}$ posibles: deben salir los números de Bell ($1, 2, 5, 15$), los órdenes parciales ($1, 3, 19, 219$) y $n!$ órdenes totales.

In [6]:
def es_equivalencia(u, r):
    return es_reflexiva(u, r) and es_simetrica(r) and es_transitiva(r)


def es_orden_parcial(u, r):
    return es_reflexiva(u, r) and es_antisimetrica(r) and es_transitiva(r)


def es_orden_total(u, r):
    return es_orden_parcial(u, r) and es_total(u, r)


MISMO_RESTO = {(a, b) for a in U_DIV for b in U_DIV if a % 3 == b % 3}
PARTES = {frozenset(c) for k in range(4) for c in itertools.combinations((1, 2, 3), k)}
INCLUSION = {(a, b) for a in PARTES for b in PARTES if a <= b}

ejemplos = [
    ("Divisibilidad", U_DIV, DIV, "orden parcial"),
    ("Menor o igual", U_DIV, MENOR_IGUAL, "orden total"),
    ("Menor que", U_DIV, MENOR, "ninguna"),
    ("Misma isla", U_MUN, MISMA_ISLA, "equivalencia"),
    ("Mismo resto (3)", U_DIV, MISMO_RESTO, "equivalencia"),
    ("Inclusión", PARTES, INCLUSION, "orden parcial"),
    ("Guagua directa", U_GUA, GUAGUA, "ninguna"),
]


def tipo(u, r):
    if es_equivalencia(u, r):
        return "equivalencia"
    if es_orden_total(u, r):
        return "orden total"
    return "orden parcial" if es_orden_parcial(u, r) else "ninguna"


for nombre, u, r, esperado in ejemplos:
    print(f"{nombre:<16} {tipo(u, r)}")
    assert tipo(u, r) == esperado


def todas_las_relaciones(u):
    pares = [(x, y) for x in u for y in u]
    for mascara in range(2 ** len(pares)):
        yield {p for i, p in enumerate(pares) if mascara >> i & 1}


print(f"\n{'n':>2} {'relaciones':>11} {'equivalencias':>14} {'órdenes parciales':>18} {'órdenes totales':>16}")
for n, esperado in zip(range(1, 5), [(1, 1, 1), (2, 3, 2), (5, 19, 6), (15, 219, 24)]):
    u = set(range(n))
    cuenta = Counter()
    for r in todas_las_relaciones(u):
        cuenta["equivalencia"] += es_equivalencia(u, r)
        if es_orden_parcial(u, r):
            cuenta["parcial"] += 1
            cuenta["total"] += es_total(u, r)
    print(f"{n:>2} {2 ** (n * n):>11} {cuenta['equivalencia']:>14} {cuenta['parcial']:>18} {cuenta['total']:>16}")
    assert (cuenta["equivalencia"], cuenta["parcial"], cuenta["total"]) == esperado

Divisibilidad    orden parcial
Menor o igual    orden total
Menor que        ninguna
Misma isla       equivalencia
Mismo resto (3)  equivalencia
Inclusión        orden parcial
Guagua directa   ninguna

 n  relaciones  equivalencias  órdenes parciales  órdenes totales
 1           2              1                  1                1
 2          16              2                  3                2
 3         512              5                 19                6
 4       65536             15                219               24


## 7. Composición de relaciones

La composición $R \circ S$ contiene los pares $(x, z)$ para los que hay un $y$ con $(x, y) \in R$ e $(y, z) \in S$ (aquí se aplica primero $R$ y luego $S$). Con las guaguas, `componer(GUAGUA, GUAGUA)` da los destinos a los que se llega cogiendo exactamente dos. Comprobar con relaciones aleatorias que la composición es asociativa, que $(R \circ S)^{-1} = S^{-1} \circ R^{-1}$ y que equivale al producto booleano de las matrices, y mostrar que no es conmutativa.

In [7]:
def componer(r, s):
    suc = sucesores(s)
    return {(x, z) for x, y in r for z in suc[y]}


def producto_booleano(a, b):
    n = len(a)
    return [[any(a[i][k] and b[k][j] for k in range(n)) for j in range(n)] for i in range(n)]


dos = componer(GUAGUA, GUAGUA)
for pueblo in sorted(U_GUA):
    print(f"{pueblo:<10} con dos guaguas llega a: {', '.join(sorted(imagen(dos, pueblo)))}")

rng = random.Random(4)
for _ in range(500):
    u = set(range(rng.randint(1, 6)))
    r, s, t = (relacion_aleatoria(u, 0.3, rng) for _ in range(3))
    assert componer(componer(r, s), t) == componer(r, componer(s, t))
    assert inversa(componer(r, s)) == componer(inversa(s), inversa(r))
    orden, mr = a_matriz(u, r)
    _, ms = a_matriz(u, s)
    assert de_matriz(orden, producto_booleano(mr, ms)) == componer(r, s)

r, s = {(1, 2)}, {(2, 3)}
print(f"\nR = {r}, S = {s}: R∘S = {componer(r, s)}, S∘R = {componer(s, r)}")
assert componer(r, s) != componer(s, r)

Agaete     con dos guaguas llega a: Agaete


Agüimes    con dos guaguas llega a: Agüimes, Las Palmas
Arucas     con dos guaguas llega a: Agaete
Gáldar     con dos guaguas llega a: Gáldar
Las Palmas con dos guaguas llega a: Agüimes, Gáldar, Las Palmas
Telde      con dos guaguas llega a: Arucas, Telde

R = {(1, 2)}, S = {(2, 3)}: R∘S = {(1, 3)}, S∘R = set()


## 8. Potencias de una relación

Definir $R^0$ como la identidad $I = \{(x, x) : x \in U\}$ y $R^n = R^{n-1} \circ R$. Mostrar a qué pueblos se llega desde Las Palmas con exactamente $n$ guaguas para $n = 0, \dots, 5$. Como en un universo finito solo hay $2^{|U|^2}$ relaciones, la sucesión de potencias acaba repitiéndose: calcular a partir de qué exponente (índice) y cada cuánto (periodo). Comprobar que $R^{a+b} = R^a \circ R^b$ y que $R$ es transitiva si y solo si $R^2 \subseteq R$.

In [8]:
def identidad(u):
    return {(x, x) for x in u}


def potencia(u, r, n):
    resultado = identidad(u)
    for _ in range(n):
        resultado = componer(resultado, r)
    return resultado


def indice_y_periodo(u, r):
    vistas = {}
    actual, n = identidad(u), 0
    while (clave := frozenset(actual)) not in vistas:
        vistas[clave] = n
        actual, n = componer(actual, r), n + 1
    return vistas[clave], n - vistas[clave]


for n in range(6):
    print(f"n = {n}: {', '.join(sorted(imagen(potencia(U_GUA, GUAGUA, n), 'Las Palmas')))}")

CICLO = {(i, (i + 1) % 4) for i in range(4)}
for nombre, u, r in [("Guaguas", U_GUA, GUAGUA), ("Ciclo de 4", set(range(4)), CICLO), ("Menor que", U_DIV, MENOR)]:
    i, p = indice_y_periodo(u, r)
    print(f"{nombre:<11} índice {i:>2}, periodo {p}")
assert indice_y_periodo(set(range(4)), CICLO) == (0, 4)
assert indice_y_periodo(U_DIV, MENOR) == (12, 1)

rng = random.Random(5)
for _ in range(300):
    u = set(range(rng.randint(1, 5)))
    r = relacion_aleatoria(u, 0.3, rng)
    a, b = rng.randint(0, 4), rng.randint(0, 4)
    assert potencia(u, r, a + b) == componer(potencia(u, r, a), potencia(u, r, b))
    assert es_transitiva(r) == (componer(r, r) <= r)
    i, p = indice_y_periodo(u, r)
    assert potencia(u, r, i) == potencia(u, r, i + p)

n = 0: Las Palmas
n = 1: Arucas, Telde
n = 2: Agüimes, Gáldar, Las Palmas
n = 3: Agaete, Arucas, Telde
n = 4: Agüimes, Gáldar, Las Palmas
n = 5: Agaete, Arucas, Telde
Guaguas     índice  4, periodo 2
Ciclo de 4  índice  0, periodo 4
Menor que   índice 12, periodo 1


## 9. Clausuras reflexiva y simétrica

La clausura de $R$ respecto a una propiedad es la menor relación que contiene a $R$ y la cumple. La reflexiva es $R \cup I$ y la simétrica, $R \cup R^{-1}$. Escribirlas y comprobarlas contra una clausura por fuerza bruta: la intersección de todas las relaciones del universo que contienen a $R$ y tienen la propiedad (con $|U| = 3$ hay como mucho $2^9$ candidatas).

In [9]:
def clausura_reflexiva(u, r):
    return r | identidad(u)


def clausura_simetrica(r):
    return r | inversa(r)


def clausura_bruta(u, r, propiedad):
    resto = [(x, y) for x in u for y in u if (x, y) not in r]
    candidatas = (r | set(c) for k in range(len(resto) + 1) for c in itertools.combinations(resto, k))
    return set.intersection(*(c for c in candidatas if propiedad(c)))


print("Trayectos de vuelta que faltan:", sorted(clausura_simetrica(GUAGUA) - GUAGUA))

rng = random.Random(6)
u = {0, 1, 2}
for _ in range(40):
    r = relacion_aleatoria(u, 0.3, rng)
    assert clausura_reflexiva(u, r) == clausura_bruta(u, r, lambda c: es_reflexiva(u, c))
    assert clausura_simetrica(r) == clausura_bruta(u, r, es_simetrica)

Trayectos de vuelta que faltan: [('Arucas', 'Las Palmas'), ('Gáldar', 'Arucas')]


## 10. Clausura transitiva: Warshall y punto fijo

Calcular la clausura transitiva $R^+$ de tres formas: con el algoritmo de Warshall sobre la matriz (el mismo de *Floyd-Warshall*, $O(|U|^3)$), por punto fijo añadiendo un paso más ($T \leftarrow T \cup T \circ R$) hasta que no cambie y por punto fijo elevando al cuadrado ($T \leftarrow T \cup T \circ T$), que dobla la longitud de los caminos en cada vuelta. Mostrar a qué pueblos se llega desde cada uno con cualquier número de guaguas, contar las vueltas de los dos puntos fijos en una cadena $0 \to 1 \to \dots \to n-1$ y comprobar con relaciones aleatorias que:

- Las tres versiones coinciden entre sí, con $R^+ = R \cup R^2 \cup \dots \cup R^{|U|}$ y con la fuerza bruta.
- La clausura transitiva de una relación simétrica es simétrica, y la de una reflexiva, reflexiva.
- La clausura transitiva de la simétrica de la reflexiva es una equivalencia.
- En cambio, la clausura simétrica de una relación transitiva puede no ser transitiva.

In [10]:
def clausura_transitiva_warshall(u, r):
    orden, m = a_matriz(u, r)
    for k in range(len(orden)):
        for fila in m:
            if fila[k]:
                for j, v in enumerate(m[k]):
                    if v:
                        fila[j] = True
    return de_matriz(orden, m)


def clausura_transitiva_lineal(r):
    t, vueltas = set(r), 0
    while (nueva := t | componer(t, r)) != t:
        t, vueltas = nueva, vueltas + 1
    return t, vueltas


def clausura_transitiva_cuadrados(r):
    t, vueltas = set(r), 0
    while (nueva := t | componer(t, t)) != t:
        t, vueltas = nueva, vueltas + 1
    return t, vueltas


alcance = clausura_transitiva_warshall(U_GUA, GUAGUA)
for pueblo in sorted(U_GUA):
    print(f"Desde {pueblo:<10} se llega a: {', '.join(sorted(imagen(alcance, pueblo)))}")

print(f"\n{'n':>4} {'vueltas T∘R':>12} {'vueltas T∘T':>12}")
for n in (2, 3, 5, 10, 50, 100):
    cadena = {(i, i + 1) for i in range(n - 1)}
    t1, v1 = clausura_transitiva_lineal(cadena)
    t2, v2 = clausura_transitiva_cuadrados(cadena)
    assert t1 == t2 == clausura_transitiva_warshall(set(range(n)), cadena)
    assert len(t1) == n * (n - 1) // 2
    print(f"{n:>4} {v1:>12} {v2:>12}")

rng = random.Random(7)
for _ in range(300):
    u = set(range(rng.randint(1, 8)))
    r = relacion_aleatoria(u, 0.2, rng)
    w = clausura_transitiva_warshall(u, r)
    assert w == clausura_transitiva_lineal(r)[0] == clausura_transitiva_cuadrados(r)[0]
    assert w == set().union(*(potencia(u, r, k) for k in range(1, len(u) + 1)))
    assert es_simetrica(clausura_transitiva_warshall(u, clausura_simetrica(r)))
    assert es_reflexiva(u, clausura_transitiva_warshall(u, clausura_reflexiva(u, r)))
    assert es_equivalencia(u, clausura_transitiva_warshall(u, clausura_simetrica(clausura_reflexiva(u, r))))
u = {0, 1, 2}
for _ in range(40):
    r = relacion_aleatoria(u, 0.3, rng)
    assert clausura_transitiva_warshall(u, r) == clausura_bruta(u, r, es_transitiva)

r = {(1, 2)}
print(f"\n{r} es transitiva, pero {clausura_simetrica(r)} no: falta {next(contraejemplos_transitiva(clausura_simetrica(r)))[::2]}")
assert es_transitiva(r) and not es_transitiva(clausura_simetrica(r))

Desde Agaete     se llega a: Agaete, Gáldar
Desde Agüimes    se llega a: Agaete, Agüimes, Arucas, Gáldar, Las Palmas, Telde
Desde Arucas     se llega a: Agaete, Gáldar
Desde Gáldar     se llega a: Agaete, Gáldar
Desde Las Palmas se llega a: Agaete, Agüimes, Arucas, Gáldar, Las Palmas, Telde
Desde Telde      se llega a: Agaete, Agüimes, Arucas, Gáldar, Las Palmas, Telde

   n  vueltas T∘R  vueltas T∘T
   2            0            0
   3            1            1
   5            3            2
  10            8            4


  50           48            6
 100           98            7



{(1, 2)} es transitiva, pero {(1, 2), (2, 1)} no: falta (1, 1)


## 11. Clases de equivalencia y conjunto cociente

La clase de $x$ es $[x] = \{y : (x, y) \in R\}$ y el conjunto cociente $U/R$ reúne las clases distintas. Calcular el cociente de «misma isla» y de «mismo resto al dividir entre 3», y el camino inverso: la relación que define una partición (dos elementos se relacionan si están en el mismo bloque). Comprobar con relaciones aleatorias que $R$ es de equivalencia si y solo si cada $x$ está en su clase y las clases forman una partición de $U$, y ver con un ejemplo por qué hace falta la primera condición.

In [11]:
def clase(r, x):
    return frozenset(imagen(r, x))


def cociente(u, r):
    return {clase(r, x) for x in u}


def es_particion(u, bloques):
    return all(bloques) and set().union(*bloques) == u and sum(map(len, bloques)) == len(u)


def relacion_de_particion(bloques):
    return {(x, y) for b in bloques for x in b for y in b}


def particion_aleatoria(u, rng):
    k = rng.randint(1, len(u))
    bloques = defaultdict(set)
    for x in u:
        bloques[rng.randrange(k)].add(x)
    return {frozenset(b) for b in bloques.values()}


def clases_forman_particion(u, r):
    return all(x in clase(r, x) for x in u) and es_particion(u, cociente(u, r))


for nombre, u, r in [("Misma isla", U_MUN, MISMA_ISLA), ("Mismo resto (3)", U_DIV, MISMO_RESTO)]:
    print(f"{nombre}:")
    for b in sorted(cociente(u, r), key=min):
        print("  {" + ", ".join(map(str, sorted(b))) + "}")
assert len(cociente(U_MUN, MISMA_ISLA)) == len(set(ISLA.values()))
assert relacion_de_particion(cociente(U_DIV, MISMO_RESTO)) == MISMO_RESTO

rng = random.Random(8)
equivalencias = 0
for _ in range(2000):
    u = set(range(rng.randint(1, 6)))
    if rng.random() < 0.5:
        r = relacion_de_particion(particion_aleatoria(u, rng))
    else:
        r = relacion_aleatoria(u, rng.random(), rng)
    assert es_equivalencia(u, r) == clases_forman_particion(u, r)
    if es_equivalencia(u, r):
        equivalencias += 1
        assert relacion_de_particion(cociente(u, r)) == r
print(f"\n{equivalencias} de 2000 relaciones aleatorias eran de equivalencia")

u, r = {1, 2}, {(1, 2), (2, 1)}
print(f"R = {r}: clases {sorted(map(set, cociente(u, r)), key=min)}, ¿partición? {es_particion(u, cociente(u, r))}, "
      f"¿equivalencia? {es_equivalencia(u, r)} (1 no está en su clase)")
assert es_particion(u, cociente(u, r)) and not es_equivalencia(u, r)

Misma isla:
  {Adeje, La Laguna, Santa Cruz}
  {Agüimes, Las Palmas de GC, Telde}
  {Arrecife}
  {Puerto del Rosario}
Mismo resto (3):
  {1, 4, 7, 10}
  {2, 5, 8, 11}
  {3, 6, 9, 12}

1158 de 2000 relaciones aleatorias eran de equivalencia
R = {(1, 2), (2, 1)}: clases [{1}, {2}], ¿partición? True, ¿equivalencia? False (1 no está en su clase)


## 12. Maximales, minimales, máximo y mínimo

En un orden parcial, $x$ es maximal si no hay ningún otro elemento por encima de él y es el máximo si todos están por debajo (minimal y mínimo, al revés). Calcularlos para la divisibilidad en $\{2, \dots, 12\}$, para los divisores de 60 y para «es prefijo de» entre palabras. Comprobar con órdenes parciales aleatorios (la clausura reflexiva y transitiva de un grafo con aristas $i \to j$, $i < j$) que siempre hay algún maximal y algún minimal, y que hay máximo si y solo si hay un único maximal, que entonces es el máximo.

In [12]:
def maximales(u, r):
    return {x for x in u if not any((x, y) in r for y in u if y != x)}


def minimales(u, r):
    return {x for x in u if not any((y, x) in r for y in u if y != x)}


def maximo(u, r):
    return next((x for x in u if all((y, x) in r for y in u)), None)


def minimo(u, r):
    return next((x for x in u if all((x, y) in r for y in u)), None)


def divisibilidad(u):
    return {(a, b) for a in u for b in u if b % a == 0}


U2 = set(range(2, 13))
D60 = {d for d in range(1, 61) if 60 % d == 0}
PALABRAS = {"mar", "mare", "marea", "mareo", "pal", "palma", "palmera"}
PREFIJO = {(a, b) for a in PALABRAS for b in PALABRAS if b.startswith(a)}

for nombre, u, r in [("Divisibilidad en 2..12", U2, divisibilidad(U2)), ("Divisores de 60", D60, divisibilidad(D60)),
                     ("Prefijos", PALABRAS, PREFIJO)]:
    assert es_orden_parcial(u, r)
    print(f"{nombre}:\n  maximales: {sorted(maximales(u, r))}\n  minimales: {sorted(minimales(u, r))}\n"
          f"  máximo: {maximo(u, r)}, mínimo: {minimo(u, r)}")

assert minimales(U2, divisibilidad(U2)) == {2, 3, 5, 7, 11}
assert maximales(U2, divisibilidad(U2)) == {7, 8, 9, 10, 11, 12}
assert (maximo(D60, divisibilidad(D60)), minimo(D60, divisibilidad(D60))) == (60, 1)


def orden_aleatorio(n, p, rng):
    u = set(range(n))
    aristas = {(i, j) for i in range(n) for j in range(i + 1, n) if rng.random() < p}
    return u, clausura_transitiva_warshall(u, clausura_reflexiva(u, aristas))


rng = random.Random(9)
for _ in range(500):
    u, r = orden_aleatorio(rng.randint(1, 8), rng.random(), rng)
    assert es_orden_parcial(u, r)
    mx, mn = maximales(u, r), minimales(u, r)
    assert mx and mn
    assert (len(mx) == 1) == (maximo(u, r) is not None)
    assert (len(mn) == 1) == (minimo(u, r) is not None)
    if len(mx) == 1:
        assert mx == {maximo(u, r)}

Divisibilidad en 2..12:
  maximales: [7, 8, 9, 10, 11, 12]
  minimales: [2, 3, 5, 7, 11]
  máximo: None, mínimo: None
Divisores de 60:
  maximales: [60]
  minimales: [1]
  máximo: 60, mínimo: 1
Prefijos:
  maximales: ['marea', 'mareo', 'palma', 'palmera']
  minimales: ['mar', 'pal']
  máximo: None, mínimo: None


## 13. Diagrama de Hasse

El diagrama de Hasse de un orden parcial dibuja solo las relaciones inmediatas: $x \prec y$ si $x < y$ y no hay ningún $z$ con $x < z < y$. Es la reducción transitiva del orden estricto $S = R \setminus I$, que se obtiene quitándole los pares alcanzables en dos pasos: $S \setminus (S \circ S)$. Obtener las aristas del diagrama de los divisores de 60, compararlo con la definición por fuerza bruta y comprobar en órdenes aleatorios que la clausura reflexiva y transitiva de las aristas recupera el orden y que no sobra ninguna. En los divisores de $n$, las aristas unen $d$ con $d \cdot p$ para cada primo $p$ con $d p \mid n$, lo que permite contarlas.

In [13]:
def diagrama_hasse(u, r):
    estricto = r - identidad(u)
    return estricto - componer(estricto, estricto)


def hasse_bruta(u, r):
    return {(x, y) for x, y in r
            if x != y and not any(z not in (x, y) and (x, z) in r and (z, y) in r for z in u)}


R60 = divisibilidad(D60)
H60 = diagrama_hasse(D60, R60)
for d in sorted(D60):
    if arriba := sorted(y for x, y in H60 if x == d):
        print(f"{d:>2} → {', '.join(map(str, arriba))}")
print(f"{len(H60)} aristas en el diagrama de los {len(R60) - len(D60)} pares del orden estricto")

PRIMOS_60 = (2, 3, 5)
assert H60 == hasse_bruta(D60, R60)
assert all(y // x in PRIMOS_60 for x, y in H60)
assert len(H60) == sum(1 for d in D60 for p in PRIMOS_60 if 60 % (d * p) == 0)

rng = random.Random(10)
for _ in range(300):
    u, r = orden_aleatorio(rng.randint(1, 8), rng.random(), rng)
    h = diagrama_hasse(u, r)
    assert h == hasse_bruta(u, r)
    assert clausura_transitiva_warshall(u, clausura_reflexiva(u, h)) == r
    for arista in h:
        assert clausura_transitiva_warshall(u, clausura_reflexiva(u, h - {arista})) != r

 1 → 2, 3, 5
 2 → 4, 6, 10
 3 → 6, 15
 4 → 12, 20
 5 → 10, 15
 6 → 12, 30
10 → 20, 30
12 → 60
15 → 30
20 → 60
30 → 60
20 aristas en el diagrama de los 42 pares del orden estricto
